<a href="https://colab.research.google.com/github/AmalMohammed95/SARA-Smart-Academic-Research-Agent/blob/main/SARA_Smart_Academic_Research_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# SARA - Smart Academic Research Agent
# Project 02 - Elicit
# Level 2 - Applied Agent

print("SARA environment is ready.")

SARA environment is ready.


In [3]:
!pip -q install requests pandas

In [4]:
import requests
import pandas as pd
import json
import sqlite3
import time

print("All basic libraries loaded successfully.")

All basic libraries loaded successfully.


In [5]:
# SARA Agent State

sara_state = {
    "research_question": "",
    "sub_questions": [],
    "search_keywords": [],
    "retrieved_papers": [],
    "selected_papers": [],
    "excluded_papers": [],
    "exclusion_reasons": {},
    "extracted_evidence": [],
    "identified_gaps": [],
    "iteration": 0,
    "stopping_reason": None,
    "execution_log": []
}

print("SARA state initialized successfully.")
print(sara_state)

SARA state initialized successfully.
{'research_question': '', 'sub_questions': [], 'search_keywords': [], 'retrieved_papers': [], 'selected_papers': [], 'excluded_papers': [], 'exclusion_reasons': {}, 'extracted_evidence': [], 'identified_gaps': [], 'iteration': 0, 'stopping_reason': None, 'execution_log': []}


In [7]:
# Step 4 - Set Research Question

research_question = "What are the recent applications of agentic AI in academic research and literature review?"

sara_state["research_question"] = research_question

sara_state["sub_questions"] = [
    "What is agentic AI in the context of academic research?",
    "How is agentic AI used in literature review workflows?",
    "What tools or frameworks are commonly used?",
    "What are the main benefits and limitations?"
]

sara_state["search_keywords"] = [
    "agentic AI academic research",
    "agentic AI literature review",
    "AI agents scholarly research",
    "autonomous agents literature review"
]

sara_state["execution_log"].append({
    "step": "research_planning",
    "status": "completed",
    "details": "Research question decomposed into sub-questions and search keywords."
})

print("Research plan created successfully.\n")

print("Research Question:")
print(sara_state["research_question"])

print("\nSub-Questions:")
for i, q in enumerate(sara_state["sub_questions"], 1):
    print(f"{i}. {q}")

print("\nSearch Keywords:")
for i, k in enumerate(sara_state["search_keywords"], 1):
    print(f"{i}. {k}")

Research plan created successfully.

Research Question:
What are the recent applications of agentic AI in academic research and literature review?

Sub-Questions:
1. What is agentic AI in the context of academic research?
2. How is agentic AI used in literature review workflows?
3. What tools or frameworks are commonly used?
4. What are the main benefits and limitations?

Search Keywords:
1. agentic AI academic research
2. agentic AI literature review
3. AI agents scholarly research
4. autonomous agents literature review


In [8]:
# Step 5 - Search OpenAlex

def search_openalex(query, per_page=5):
    url = "https://api.openalex.org/works"

    params = {
        "search": query,
        "filter": "from_publication_date:2021-01-01,language:en",
        "per-page": per_page
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    papers = []

    for work in data.get("results", []):
        paper = {
            "title": work.get("title"),
            "year": work.get("publication_year"),
            "doi": work.get("doi"),
            "openalex_id": work.get("id"),
            "cited_by_count": work.get("cited_by_count", 0)
        }
        papers.append(paper)

    return papers


# Test the search tool
test_query = sara_state["search_keywords"][0]

papers = search_openalex(test_query)

print(f"Search query: {test_query}")
print(f"Papers retrieved: {len(papers)}\n")

for i, paper in enumerate(papers, 1):
    print(f"{i}. {paper['title']}")
    print(f"   Year: {paper['year']}")
    print(f"   DOI: {paper['doi']}")
    print(f"   Citations: {paper['cited_by_count']}")
    print()

Search query: agentic AI academic research
Papers retrieved: 5

1. Opinion Paper: “So what if ChatGPT wrote it?” Multidisciplinary perspectives on opportunities, challenges and implications of generative conversational AI for research, practice and policy
   Year: 2023
   DOI: https://doi.org/10.1016/j.ijinfomgt.2023.102642
   Citations: 4332

2. Conceptualizing AI literacy: An exploratory review
   Year: 2021
   DOI: https://doi.org/10.1016/j.caeai.2021.100041
   Citations: 1909

3. Performance of ChatGPT on USMLE: Potential for AI-assisted medical education using large language models
   Year: 2023
   DOI: https://doi.org/10.1371/journal.pdig.0000198
   Citations: 3846

4. Generative AI
   Year: 2023
   DOI: https://doi.org/10.1007/s12599-023-00834-7
   Citations: 1369

5. Students’ voices on generative AI: perceptions, benefits, and challenges in higher education
   Year: 2023
   DOI: https://doi.org/10.1186/s41239-023-00411-8
   Citations: 2213



In [9]:
# Test OpenAlex connection

test_results = search_openalex(
    "agentic AI academic research",
    per_page=3
)

print("OpenAlex connection successful.")
print("Papers retrieved:", len(test_results))

for i, paper in enumerate(test_results, 1):
    print(f"{i}. {paper['title']}")

OpenAlex connection successful.
Papers retrieved: 3
1. Opinion Paper: “So what if ChatGPT wrote it?” Multidisciplinary perspectives on opportunities, challenges and implications of generative conversational AI for research, practice and policy
2. Conceptualizing AI literacy: An exploratory review
3. Performance of ChatGPT on USMLE: Potential for AI-assisted medical education using large language models


In [11]:
# Step 6 - Search using all SARA keywords

all_retrieved_papers = []

for query in sara_state["search_keywords"]:
    print(f"Searching: {query}")

    results = search_openalex(query, per_page=10)

    for paper in results:
        paper["search_query"] = query
        all_retrieved_papers.append(paper)

    print(f"Retrieved: {len(results)} papers\n")


# Remove duplicates using DOI or OpenAlex ID
unique_papers = {}

for paper in all_retrieved_papers:
    key = paper["doi"] if paper["doi"] else paper["openalex_id"]

    if key not in unique_papers:
        unique_papers[key] = paper


sara_state["retrieved_papers"] = list(unique_papers.values())

sara_state["execution_log"].append({
    "step": "academic_search",
    "status": "completed",
    "details": f"{len(sara_state['retrieved_papers'])} unique papers retrieved from OpenAlex."
})


print("Search completed.")
print(f"Total retrieved before deduplication: {len(all_retrieved_papers)}")
print(f"Unique papers after deduplication: {len(sara_state['retrieved_papers'])}")

Searching: agentic AI academic research
Retrieved: 10 papers

Searching: agentic AI literature review
Retrieved: 10 papers

Searching: AI agents scholarly research
Retrieved: 10 papers

Searching: autonomous agents literature review
Retrieved: 10 papers

Search completed.
Total retrieved before deduplication: 40
Unique papers after deduplication: 32


In [12]:
# Step 7 - Paper Screening

def screen_paper(paper, research_question):
    title = (paper.get("title") or "").lower()

    # Basic relevance terms
    relevant_terms = [
        "agentic",
        "agent",
        "academic research",
        "literature review",
        "research assistant",
        "scholarly",
        "systematic review"
    ]

    # Check relevance from title
    relevance = any(term in title for term in relevant_terms)

    # Inclusion / exclusion decision
    if not relevance:
        return "exclude", "Not sufficiently relevant to the research question"

    if paper.get("year") is None or paper.get("year") < 2021:
        return "exclude", "Outside the allowed publication date range"

    if not paper.get("doi") and not paper.get("openalex_id"):
        return "exclude", "Missing verifiable persistent identifier"

    return "include", "Relevant to research question and meets basic screening criteria"


# Reset screening results
sara_state["selected_papers"] = []
sara_state["excluded_papers"] = []
sara_state["exclusion_reasons"] = {}


# Screen all retrieved papers
for paper in sara_state["retrieved_papers"]:

    decision, reason = screen_paper(
        paper,
        sara_state["research_question"]
    )

    if decision == "include":
        sara_state["selected_papers"].append(paper)

    else:
        sara_state["excluded_papers"].append(paper)

        key = paper["doi"] if paper["doi"] else paper["openalex_id"]
        sara_state["exclusion_reasons"][key] = reason


# Add to execution log
sara_state["execution_log"].append({
    "step": "paper_screening",
    "status": "completed",
    "details": {
        "retrieved": len(sara_state["retrieved_papers"]),
        "included": len(sara_state["selected_papers"]),
        "excluded": len(sara_state["excluded_papers"])
    }
})


print("Paper screening completed.\n")

print("Total retrieved:", len(sara_state["retrieved_papers"]))
print("Included papers:", len(sara_state["selected_papers"]))
print("Excluded papers:", len(sara_state["excluded_papers"]))

Paper screening completed.

Total retrieved: 32
Included papers: 14
Excluded papers: 18


In [13]:
# Step 8 - Convert OpenAlex inverted abstract to normal text

def reconstruct_abstract(inverted_index):
    if not inverted_index:
        return ""

    words = []

    for word, positions in inverted_index.items():
        for position in positions:
            words.append((position, word))

    words.sort(key=lambda x: x[0])

    return " ".join(word for _, word in words)


print("Abstract reconstruction function is ready.")

Abstract reconstruction function is ready.


In [15]:
# Test abstract reconstruction on a paper with an available abstract

test_paper = None

for paper in sara_state["retrieved_papers"]:
    if paper.get("abstract_inverted_index"):
        test_paper = paper
        break

if test_paper:
    abstract = reconstruct_abstract(
        test_paper.get("abstract_inverted_index")
    )

    print("Title:", test_paper.get("title"))
    print("Abstract available:", bool(abstract))
    print("\nAbstract preview:")
    print(abstract[:500])
else:
    print("No paper with an abstract was found.")

No paper with an abstract was found.


In [16]:
# Step 10 - Improved OpenAlex search with abstracts

def search_openalex(query, per_page=10):
    url = "https://api.openalex.org/works"

    params = {
        "search": query,
        "filter": "from_publication_date:2021-01-01,language:en",
        "per-page": per_page
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    papers = []

    for work in data.get("results", []):

        abstract = reconstruct_abstract(
            work.get("abstract_inverted_index")
        )

        paper = {
            "title": work.get("title"),
            "abstract": abstract,
            "year": work.get("publication_year"),
            "doi": work.get("doi"),
            "openalex_id": work.get("id"),
            "cited_by_count": work.get("cited_by_count", 0)
        }

        papers.append(paper)

    return papers


print("Improved OpenAlex search function is ready.")

Improved OpenAlex search function is ready.


In [17]:
# Step 11 - Re-run search with abstracts

all_retrieved_papers = []

for query in sara_state["search_keywords"]:
    print(f"Searching: {query}")

    results = search_openalex(query, per_page=10)

    for paper in results:
        paper["search_query"] = query
        all_retrieved_papers.append(paper)

    print(f"Retrieved: {len(results)} papers\n")


# Remove duplicate papers
unique_papers = {}

for paper in all_retrieved_papers:
    key = paper["doi"] if paper["doi"] else paper["openalex_id"]

    if key not in unique_papers:
        unique_papers[key] = paper


# Replace old retrieved papers with improved records
sara_state["retrieved_papers"] = list(unique_papers.values())


# Count papers that have abstracts
papers_with_abstract = sum(
    1 for paper in sara_state["retrieved_papers"]
    if paper.get("abstract")
)


sara_state["execution_log"].append({
    "step": "improved_academic_search",
    "status": "completed",
    "details": {
        "total_unique_papers": len(sara_state["retrieved_papers"]),
        "papers_with_abstract": papers_with_abstract
    }
})


print("Improved search completed.")
print("Unique papers:", len(sara_state["retrieved_papers"]))
print("Papers with abstracts:", papers_with_abstract)

Searching: agentic AI academic research
Retrieved: 10 papers

Searching: agentic AI literature review
Retrieved: 10 papers

Searching: AI agents scholarly research
Retrieved: 10 papers

Searching: autonomous agents literature review
Retrieved: 10 papers

Improved search completed.
Unique papers: 32
Papers with abstracts: 31


In [18]:
# Step 12 - Improved Screening using Title + Abstract

def improved_screen_paper(paper):
    title = (paper.get("title") or "").lower()
    abstract = (paper.get("abstract") or "").lower()

    text = title + " " + abstract

    # Strong relevance terms
    agentic_terms = [
        "agentic ai",
        "autonomous agent",
        "autonomous agents",
        "llm agent",
        "llm agents",
        "large language model agent",
        "large language model agents",
        "multi-agent",
        "multi agent"
    ]

    # Academic research / literature review context terms
    research_terms = [
        "academic research",
        "literature review",
        "systematic review",
        "scholarly research",
        "research workflow",
        "research assistant",
        "evidence synthesis",
        "paper screening",
        "academic search"
    ]

    has_agentic_term = any(term in text for term in agentic_terms)
    has_research_term = any(term in text for term in research_terms)

    # Date rule
    if paper.get("year") is None or paper.get("year") < 2021:
        return "exclude", "Outside the allowed publication date range"

    # Metadata verification rule
    if not paper.get("doi") and not paper.get("openalex_id"):
        return "exclude", "Missing verifiable persistent identifier"

    # Abstract availability rule
    if not paper.get("abstract"):
        return "exclude", "No available abstract"

    # Main relevance rule
    if has_agentic_term and has_research_term:
        return "include", "Relevant agentic AI paper in an academic research context"

    return "exclude", "Does not sufficiently match both agentic AI and academic research context"


# Reset previous screening
sara_state["selected_papers"] = []
sara_state["excluded_papers"] = []
sara_state["exclusion_reasons"] = {}


for paper in sara_state["retrieved_papers"]:

    decision, reason = improved_screen_paper(paper)

    if decision == "include":
        sara_state["selected_papers"].append(paper)

    else:
        sara_state["excluded_papers"].append(paper)

        key = paper["doi"] if paper["doi"] else paper["openalex_id"]
        sara_state["exclusion_reasons"][key] = reason


sara_state["execution_log"].append({
    "step": "improved_paper_screening",
    "status": "completed",
    "details": {
        "retrieved": len(sara_state["retrieved_papers"]),
        "included": len(sara_state["selected_papers"]),
        "excluded": len(sara_state["excluded_papers"])
    }
})


print("Improved screening completed.\n")
print("Total retrieved:", len(sara_state["retrieved_papers"]))
print("Included papers:", len(sara_state["selected_papers"]))
print("Excluded papers:", len(sara_state["excluded_papers"]))

Improved screening completed.

Total retrieved: 32
Included papers: 3
Excluded papers: 29


In [21]:
# SARA Baseline - Single Search Workflow

def run_sara_baseline(research_question, top_k=10):
    """
    Simple non-agentic baseline:
    one fixed search query -> one OpenAlex search -> returned papers.

    No replanning, memory, critic, or autonomous agent loop.
    """

    baseline_query = "agentic AI academic research literature review"

    results = search_openalex(
        baseline_query,
        per_page=top_k
    )

    baseline_result = {
        "research_question": research_question,
        "search_query": baseline_query,
        "retrieved_count": len(results),
        "papers": results,
        "replanning": False,
        "memory": False,
        "critic": False,
        "agent_loop": False
    }

    return baseline_result


baseline_result = run_sara_baseline(
    sara_state["research_question"],
    top_k=10
)

print("=== SARA BASELINE ===")
print("Research Question:", baseline_result["research_question"])
print("Search Query:", baseline_result["search_query"])
print("Retrieved Papers:", baseline_result["retrieved_count"])

print("\nAgentic Capabilities:")
print("Replanning:", baseline_result["replanning"])
print("Memory:", baseline_result["memory"])
print("Critic:", baseline_result["critic"])
print("Agent Loop:", baseline_result["agent_loop"])

print("\nTop Retrieved Papers:")
for i, paper in enumerate(baseline_result["papers"], 1):
    print(f"{i}. {paper['title']}")

=== SARA BASELINE ===
Research Question: What are the recent applications of agentic AI in academic research and literature review?
Search Query: agentic AI academic research literature review
Retrieved Papers: 10

Agentic Capabilities:
Replanning: False
Memory: False
Critic: False
Agent Loop: False

Top Retrieved Papers:
1. Conceptualizing AI literacy: An exploratory review
2. Opinion Paper: “So what if ChatGPT wrote it?” Multidisciplinary perspectives on opportunities, challenges and implications of generative conversational AI for research, practice and policy
3. AI in marketing, consumer research and psychology: A systematic literature review and research agenda
4. ChatGPT Utility in Healthcare Education, Research, and Practice: Systematic Review on the Promising Perspectives and Valid Concerns
5. AI literacy in K-12: a systematic literature review
6. A Review of Artificial Intelligence (AI) in Education from 2010 to 2020
7. Artificial intelligence in information systems research: 

In [22]:
# Baseline Observation

baseline_observation = {
    "retrieved_papers": baseline_result["retrieved_count"],
    "search_queries_used": 1,
    "automatic_replanning": False,
    "evidence_sufficiency_check": False,
    "memory_used": False,
    "critic_used": False,
    "observation": (
        "The baseline successfully retrieved academic papers using a single "
        "fixed query, but the returned results include broad AI-related studies. "
        "The baseline cannot assess evidence sufficiency, revise its search "
        "strategy, or recover through replanning."
    )
}

print("=== BASELINE OBSERVATION ===")
for key, value in baseline_observation.items():
    print(f"{key}: {value}")

=== BASELINE OBSERVATION ===
retrieved_papers: 10
search_queries_used: 1
automatic_replanning: False
evidence_sufficiency_check: False
memory_used: False
critic_used: False
observation: The baseline successfully retrieved academic papers using a single fixed query, but the returned results include broad AI-related studies. The baseline cannot assess evidence sufficiency, revise its search strategy, or recover through replanning.
